# incremental_sql_demo — Pipeline Walkthrough

**Observe how OpenMedallion handles growing datasets — run the pipeline twice.**

| Run | What loads |
|-----|------------|
| Run 1 | Full load — 5 orders, 3 customers |
| Run 2 | Delta only — 2 new orders + Charlie's tier update |

Run cells top to bottom.

## Setup — navigate to example root

In [ ]:
import os, sqlite3
from pathlib import Path

# Two levels up: ipynb/ → retail/ → incremental_sql_demo/
example_root = Path(os.getcwd()).parent.parent
os.chdir(example_root)
print(f'Working directory: {os.getcwd()}')

## Setup — create SQLite database

Creates `data/retail.db` with 5 orders and 3 customers.

In [ ]:
import polars as pl

DB = Path('data/retail.db')
DB.parent.mkdir(parents=True, exist_ok=True)

con = sqlite3.connect(DB)
cur = con.cursor()

cur.execute("""
    CREATE TABLE IF NOT EXISTS orders (
        order_id    INTEGER PRIMARY KEY,
        customer_id INTEGER NOT NULL,
        amount      REAL    NOT NULL,
        created_at  TEXT    NOT NULL
    )
""")
cur.execute("""
    CREATE TABLE IF NOT EXISTS customers (
        customer_id INTEGER PRIMARY KEY,
        name        TEXT NOT NULL,
        email       TEXT NOT NULL,
        tier        TEXT NOT NULL
    )
""")
cur.executemany('INSERT OR REPLACE INTO orders VALUES (?,?,?,?)', [
    (1, 101, 150.00, '2024-01-05'),
    (2, 102,  45.50, '2024-01-06'),
    (3, 101, 300.00, '2024-01-07'),
    (4, 103,  80.00, '2024-01-08'),
    (5, 102, 200.00, '2024-01-09'),
])
cur.executemany('INSERT OR REPLACE INTO customers VALUES (?,?,?,?)', [
    (101, 'Alice',   'alice@example.com',   'gold'),
    (102, 'Bob',     'bob@example.com',     'silver'),
    (103, 'Charlie', 'charlie@example.com', 'bronze'),
])
con.commit()
con.close()
print(f'✅  Database seeded at {DB}  (5 orders, 3 customers)')

---
## Run 1 — Full Bronze Load

dlt reads all orders (cursor starts at `2024-01-01`) and all customers (full scan + merge).

In [ ]:
!medallion run retail --layer bronze

In [ ]:
# Show what landed in bronze
bronze_dir = Path('data/bronze')
for f in sorted(bronze_dir.glob('*.parquet')):
    print(f'── {f.name} ──')
    print(pl.read_parquet(f))
    print()

## Run 1 — Silver + Gold

In [ ]:
!medallion run retail

In [ ]:
gold = Path('data/gold/retail')

print('── customer_summary (Run 1) ──')
run1_summary = pl.read_parquet(gold / 'customer_summary.parquet').sort('customer_id')
print(run1_summary)

print('\n── pipeline_totals (Run 1) ──')
print(pl.read_parquet(gold / 'pipeline_totals.parquet'))

---
## Delta: Add New Data

Inserts 2 new orders (Feb 2024) and promotes Charlie from `bronze` to `platinum`.

On the next bronze run:
- **orders** (append): dlt reads only rows where `created_at > 2024-01-09` → 2 new rows
- **customers** (merge): dlt upserts on `customer_id` → Charlie's tier updated

In [ ]:
con = sqlite3.connect('data/retail.db')
cur = con.cursor()

cur.executemany('INSERT OR REPLACE INTO orders VALUES (?,?,?,?)', [
    (6, 101,  90.00, '2024-02-01'),
    (7, 103, 450.00, '2024-02-02'),
])
cur.execute('UPDATE customers SET tier=? WHERE customer_id=?', ('platinum', 103))

con.commit()
con.close()
print('✅  Added 2 new orders (IDs 6, 7) and promoted Charlie → platinum')

## Run 2 — Delta Bronze Load (incremental)

Watch the output: dlt should load **only 2 rows** for orders, not 7.

In [ ]:
!medallion run retail --layer bronze

## Run 2 — Silver + Gold

In [ ]:
!medallion run retail

In [ ]:
print('── customer_summary comparison ──')
run2_summary = pl.read_parquet(gold / 'customer_summary.parquet').sort('customer_id')

diff = run1_summary.join(run2_summary, on='customer_id', suffix='_run2')
print(diff)

print('\n── pipeline_totals (Run 2) ──')
print(pl.read_parquet(gold / 'pipeline_totals.parquet'))

print('\nKey observations:')
print('  • Charlie (103): total_orders went from 1 → 2, total_spent grew by $450')
print('  • Alice  (101): gained order #6 ($90)')
print('  • Bob    (102): unchanged — no new orders')

## How Cursor State Is Stored

dlt saves the last-seen `created_at` value so the next run knows where to continue:

In [ ]:
import os
state_dir = Path('data/bronze')
dlt_dirs = list(state_dir.rglob('_dlt_loads'))
if dlt_dirs:
    print('dlt state directories found:')
    for d in dlt_dirs:
        print(f'  {d}')
    print('\nDelete these to force a full reload on next run.')
else:
    print('No dlt state directories found yet.')

## Things to Try

- **Force a full reload**: delete `data/bronze/` and re-run bronze
- **Add another delta**: extend `add_delta.py` with order ID 8 and re-run
- **Change initial_value**: edit `backend/bronze.yaml` → `initial_value: '2024-01-07'` and reload to see partial data